In [8]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os

# Impostazioni iniziali
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Funzioni di supporto
def load_image(image_path):
    return cv2.imread(image_path)

def load_yolo_model():
    return YOLO('yolov8/yolov8m_olive_trained.pt')

def run_yolo_inference(model, image_path, conf=0.5):
    return model.predict(source=image_path, conf=conf)

# Funzione per aumentare saturazione e contrasto dell'immagine
def enhance_image(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv[..., 1] = cv2.multiply(hsv[..., 1], 2)  # Aumenta la saturazione
    hsv[..., 2] = cv2.multiply(hsv[..., 2], 1.5)  # Aumenta il contrasto
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# Funzione per stimare il weight basato sull'area dell'ellisse (bounding box)
def estimate_weight_from_bbox_ellipse(width, height, weight_per_pixel=0.00178, max_weight=10):
    area_ellipse = np.pi * width * height
    weight = area_ellipse * weight_per_pixel
    return min(weight, max_weight)

# Funzione per disegnare bounding boxes e confidenza
def draw_boxes_with_confidence(image, results):
    total_olives = 0
    img_with_boxes = image.copy()
    for result in results:
        boxes = result.boxes.xyxy
        confs = result.boxes.conf
        total_olives += len(boxes)
        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box[:4])
            cv2.rectangle(img_with_boxes, (x1, y1), (x2, y2), (255, 0, 0), 2)
            label = f'{conf:.2f}'
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            label_w, label_h = label_size
            cv2.rectangle(img_with_boxes, (x1, y1 - label_h - 10), (x1 + label_w, y1), (255, 0, 0), -1)
            cv2.putText(img_with_boxes, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return img_with_boxes, total_olives

# Funzione per disegnare ellissi e stimare weight con soglia massima
def draw_ellipses_and_weight(image, results):
    img_with_ellipses = image.copy()
    total_weight = 0
    weights = []

    for result in results:
        boxes = result.boxes.xyxy
        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            width = x2 - x1
            height = y2 - y1

            # Stima del weight usando l'area dell'ellisse
            weight = estimate_weight_from_bbox_ellipse(width, height)
            total_weight += weight
            weights.append(weight)

            # Disegna l'ellisse
            center = (x1 + width // 2, y1 + height // 2)
            axes = (width // 2, height // 2)
            cv2.ellipse(img_with_ellipses, center, axes, 0, 0, 360, (0, 0, 255), 2)

            # Etichetta con il weight stimato
            label = f"{weight:.2f} g"
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            label_w, label_h = label_size
            cv2.rectangle(img_with_ellipses, (x1, y1 - label_h - 10), (x1 + label_w, y1), (0, 0, 255), -1)
            cv2.putText(img_with_ellipses, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    return img_with_ellipses, total_weight, weights

def create_mask_with_olive_colors(image, results):

    img_with_mask_colors = image.copy()
    hsv_image = cv2.cvtColor(img_with_mask_colors, cv2.COLOR_BGR2HSV)

    for result in results:
        boxes = result.boxes.xyxy
        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            width = x2 - x1
            height = y2 - y1
            center = (x1 + width // 2, y1 + height // 2)
            axes = (width // 2, height // 2)

            # Estrai l'area della bounding box
            roi = hsv_image[y1:y2, x1:x2]
            
            # Calcola il colore medio (Hue, Saturation, Value)
            mean_color = cv2.mean(roi)[:3]  # Solo i primi 3 canali (H, S, V)

            # Converti il colore medio in BGR per il riempimento
            mean_color_bgr = cv2.cvtColor(np.uint8([[mean_color]]), cv2.COLOR_HSV2BGR)[0][0]

            # Disegna l'ellisse con il colore medio
            cv2.ellipse(img_with_mask_colors, center, axes, 0, 0, 360, mean_color_bgr.tolist(), -1)

            # Disegna il contorno nero attorno all'ellisse
            cv2.ellipse(img_with_mask_colors, center, axes, 0, 0, 360, (0, 0, 0), 2)

    return img_with_mask_colors


# Funzione per classificare in scala di grigi le olive
def classify_olive_by_gray_intensity(gray_value):
    if gray_value <= 120:  # Valori di grigio alti -> Olives mature (nere)
        return "Ripe", (0, 200, 0)  # Verde
    elif 120 <= gray_value < 180:  # Valori di grigio intermedi -> Olives unripe (verdi)
        return "Unripe", (0, 165, 255)  # Arancione
    else:  # Valori di grigio bassi -> Olives semi mature (gialle)
        return "Partially ripe", (0, 255, 255)  # Giallo


# Funzione per riempire le olive con il colore del contorno
def classify_and_fill_colored_contours(image, gray_image, results):
    img_with_filled_ellipses = image.copy()
    mature_count, poco_mature_count, non_mature_count = 0, 0, 0
    
    for result in results:
        boxes = result.boxes.xyxy
        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            width = x2 - x1
            height = y2 - y1
            center = (x1 + width // 2, y1 + height // 2)
            axes = (width // 2, height // 2)

            # Extract the average grayscale intensity inside the ellipse
            mask = np.zeros(gray_image.shape, dtype=np.uint8)
            cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
            mean_gray_value = cv2.mean(gray_image, mask=mask)[0]

            # Classify the olive based on the grayscale intensity
            maturity_status, fill_color = classify_olive_by_gray_intensity(mean_gray_value)

            # Riempi l'ellisse con il colore
            cv2.ellipse(img_with_filled_ellipses, center, axes, 0, 0, 360, fill_color,-2)

            # Count the olives by maturity
            if maturity_status == 'Ripe':
                mature_count += 1
            elif maturity_status == 'Partially ripe':
                poco_mature_count += 1
            else:
                non_mature_count += 1

    return img_with_filled_ellipses, mature_count, poco_mature_count, non_mature_count


# Funzione principale per processare e visualizzare
def process_and_subplot_with_maturity(folder_path, upload_path):
    model = load_yolo_model()

    results_folder = os.path.join(folder_path, 'results')
    if not os.path.exists(results_folder):
        os.makedirs(results_folder)

    plots_folder = os.path.join(results_folder, 'plots')
    if not os.path.exists(plots_folder):
        os.makedirs(plots_folder)

    images_original = []
    images_boxes = []
    images_ellipses = []
    images_mask_colors = []
    images_maturity = []
    weights_all_images = []

    mature_total, poco_mature_total, non_mature_total = 0, 0, 0
    total_olives_sum = 0
    total_weight_sum = 0

    for filename in os.listdir(upload_path):
        if filename.endswith('.jpg') or filename.endswith('.png'):  
            image_path = os.path.join(upload_path, filename)
            print(f"Processando {image_path}...")

            image = load_image(image_path)
            enhanced_image = enhance_image(image)
            results = run_yolo_inference(model, image_path, conf=0.5)

            img_boxes_conf, total_olives = draw_boxes_with_confidence(image, results)
            img_ellipses, total_weight, weights = draw_ellipses_and_weight(image, results)
            mask_colors = create_mask_with_olive_colors(enhanced_image, results)

            gray_image = cv2.cvtColor(enhanced_image, cv2.COLOR_BGR2GRAY)
            img_maturity, mature_count, poco_mature_count, non_mature_count = classify_and_fill_colored_contours(mask_colors, gray_image, results)

            # Crea la cartella specifica per ciascuna immagine nella cartella 'results'
            image_folder = os.path.join(results_folder, os.path.splitext(filename)[0])
            if not os.path.exists(image_folder):
                os.makedirs(image_folder)

            # Salva i risultati nella sottocartella
            cv2.imwrite(os.path.join(image_folder, f'{filename}_original.jpg'), image)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_boxes.jpg'), img_boxes_conf)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_ellipses.jpg'), img_ellipses)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_mask_colors.jpg'), mask_colors)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_maturity.jpg'), img_maturity)

            images_original.append(image)
            images_boxes.append((img_boxes_conf, total_olives))  # Include il numero di olive
            images_ellipses.append(img_ellipses)
            images_mask_colors.append(mask_colors)
            images_maturity.append(img_maturity)

            # Aggiornamenti dei conteggi e pesi
            mature_total += mature_count
            poco_mature_total += poco_mature_count
            non_mature_total += non_mature_count
            total_olives_sum += total_olives
            total_weight_sum += total_weight
            weights_all_images.extend(weights)

            print(f"Completato {filename}")

    # Creazione dei subplot
    num_images = len(images_original)
    fig, axs = plt.subplots(5, num_images, figsize=(5 * num_images, 30))

    for i in range(num_images):
        axs[0, i].imshow(cv2.cvtColor(images_original[i], cv2.COLOR_BGR2RGB))
        axs[0, i].set_title(f"Originale")
        axs[0, i].axis('off')

        axs[1, i].imshow(cv2.cvtColor(images_boxes[i][0], cv2.COLOR_BGR2RGB))
        axs[1, i].set_title(f"Olives rilevate (Olives: {images_boxes[i][1]})")  # Visualizza il numero di olive
        axs[1, i].axis('off')

        axs[2, i].imshow(cv2.cvtColor(images_ellipses[i], cv2.COLOR_BGR2RGB))
        axs[2, i].set_title(f"Olives con relativo weight")
        axs[2, i].axis('off')

        axs[3, i].imshow(cv2.cvtColor(images_mask_colors[i], cv2.COLOR_BGR2RGB))
        axs[3, i].set_title(f"Maschera dei colori medi")
        axs[3, i].axis('off')

        axs[4, i].imshow(cv2.cvtColor(images_maturity[i], cv2.COLOR_BGR2RGB))
        axs[4, i].set_title(f"Maturity level")
        axs[4, i].axis('off')

    plt.tight_layout()
    plt.show()

    # Output del numero totale di olive e weight stimato
    total_weight_average = total_olives_sum * 5  # Weight medio stimato (5 grammi per oliva)
    print(f"Numero totale di olive rilevate: {total_olives_sum}")
    print(f"Weight totale medio stimato: {total_weight_average:.2f} grammi")
    print(f"Weight totale approssimativo delle olive: {total_weight_sum:.2f} grammi")

    # Chart of the olive distribution by maturity
    categories = ['Mature', 'Partially ripe', 'Non mature']
    counts = [mature_total, poco_mature_total, non_mature_total]
    colors = ['lightgreen', 'yellow', 'orange']

    plt.figure(figsize=(8, 6))
    bars = plt.bar(categories, counts, color=colors, edgecolor='black')

    # Aggiungi il numero di olive sopra ciascuna barra
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{count}', ha='center', fontsize=10, color='black')

    plt.title('Olive distribution by maturity')
    plt.xlabel('Maturity level')
    plt.ylabel('Numero di olive')
    plt.savefig(os.path.join(plots_folder, 'olive_maturity_distribution.png'))  # Salva il grafico della distribuzione
    plt.show()


    # Istogramma della distribuzione dei pesi delle olive
    plt.figure(figsize=(8, 6))
    weights_hist, bins, patches = plt.hist(weights_all_images, bins=10, color='green', edgecolor='black')

    # Aggiungi il numero di olive sopra ciascuna barra dell'istogramma
    for i in range(len(patches)):
        height = weights_hist[i]
        plt.text(patches[i].get_x() + patches[i].get_width() / 2, height + 0.2,  # Aggiungi un piccolo offset per evitare la sovrapposizione
                f'{int(height)}', ha='center', va='bottom', fontsize=10, color='black')

    plt.title('Distribution del weight delle olive')
    plt.xlabel('Weight in grammi')
    plt.ylabel('Numero di olive')
    plt.savefig(os.path.join(plots_folder, 'olive_weight_distribution.png'))  # Salva il grafico della distribuzione dei pesi
    plt.show()

# Esegui la funzione
folder_path = r'C:\Users\Bartolo\OlivesDetectionProject\backend'
upload_path = os.path.join(folder_path, 'upload')
process_and_subplot_with_maturity(folder_path, upload_path)


Processando C:\Users\Bartolo\OlivesDetectionProject\backend\upload\09b352d3-0f2a-40df-809b-6fc2f154a8ed_jpg.rf.7a9173fbc3ca414a944e3a791f06cbbe.jpg...

image 1/1 C:\Users\Bartolo\OlivesDetectionProject\backend\upload\09b352d3-0f2a-40df-809b-6fc2f154a8ed_jpg.rf.7a9173fbc3ca414a944e3a791f06cbbe.jpg: 640x640 9 olives, 415.7ms
Speed: 0.0ms preprocess, 415.7ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Completato 09b352d3-0f2a-40df-809b-6fc2f154a8ed_jpg.rf.7a9173fbc3ca414a944e3a791f06cbbe.jpg
Processando C:\Users\Bartolo\OlivesDetectionProject\backend\upload\5a451135-4046-4b63-bbc8-94a81cbf933d_jpg.rf.9332cd80dd9f88c9774196fcf37635bc.jpg...

image 1/1 C:\Users\Bartolo\OlivesDetectionProject\backend\upload\5a451135-4046-4b63-bbc8-94a81cbf933d_jpg.rf.9332cd80dd9f88c9774196fcf37635bc.jpg: 640x640 16 olives, 328.3ms
Speed: 0.0ms preprocess, 328.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Completato 5a451135-4046-4b63-bbc8-94a81cbf933d_jpg.rf.9332cd

C:\Users\Bartolo\AppData\Local\Temp\ipykernel_24392\1005383286.py:254: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Bartolo\AppData\Local\Temp\ipykernel_24392\1005383286.py:279: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Numero totale di olive rilevate: 154
Weight totale medio stimato: 770.00 grammi
Weight totale approssimativo delle olive: 770.40 grammi


C:\Users\Bartolo\AppData\Local\Temp\ipykernel_24392\1005383286.py:296: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [1]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Disabilita backend interattivo per evitare l'uso di Tkinter
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os

# Impostazioni iniziali
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Funzioni di supporto
def load_image(image_path):
    return cv2.imread(image_path)

def load_yolo_model():
    return YOLO('yolov8/yolov8m_olive_trained.pt')

def run_yolo_inference(model, image_path, conf=0.5):
    return model.predict(source=image_path, conf=conf)

# Funzione per aumentare saturazione e contrasto dell'immagine
def enhance_image(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv[..., 1] = cv2.multiply(hsv[..., 1], 2)  # Aumenta la saturazione
    hsv[..., 2] = cv2.multiply(hsv[..., 2], 1.5)  # Aumenta il contrasto
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# Funzione per stimare il weight basato sull'area dell'ellisse (bounding box)
def estimate_weight_from_bbox_ellipse(width, height, weight_per_pixel=0.00178, max_weight=10):
    area_ellipse = np.pi * width * height
    weight = area_ellipse * weight_per_pixel
    return min(weight, max_weight)

# Funzione per disegnare bounding boxes e confidenza
def draw_boxes_with_confidence(image, results):
    total_olives = 0
    img_with_boxes = image.copy()
    for result in results:
        boxes = result.boxes.xyxy
        confs = result.boxes.conf
        total_olives += len(boxes)
        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box[:4])
            cv2.rectangle(img_with_boxes, (x1, y1), (x2, y2), (255, 0, 0), 2)
            label = f'{conf:.2f}'
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            label_w, label_h = label_size
            cv2.rectangle(img_with_boxes, (x1, y1 - label_h - 10), (x1 + label_w, y1), (255, 0, 0), -1)
            cv2.putText(img_with_boxes, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return img_with_boxes, total_olives

# Funzione per disegnare ellissi e stimare weight con soglia massima
def draw_ellipses_and_weight(image, results):
    img_with_ellipses = image.copy()
    total_weight = 0
    weights = []

    for result in results:
        boxes = result.boxes.xyxy
        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            width = x2 - x1
            height = y2 - y1

            # Stima del weight usando l'area dell'ellisse
            weight = estimate_weight_from_bbox_ellipse(width, height)
            total_weight += weight
            weights.append(weight)

            # Disegna l'ellisse
            center = (x1 + width // 2, y1 + height // 2)
            axes = (width // 2, height // 2)
            cv2.ellipse(img_with_ellipses, center, axes, 0, 0, 360, (0, 0, 255), 2)

            # Etichetta con il weight stimato
            label = f"{weight:.2f} g"
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            label_w, label_h = label_size
            cv2.rectangle(img_with_ellipses, (x1, y1 - label_h - 10), (x1 + label_w, y1), (0, 0, 255), -1)
            cv2.putText(img_with_ellipses, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    return img_with_ellipses, total_weight, weights

def create_mask_with_olive_colors(image, results):

    img_with_mask_colors = image.copy()
    hsv_image = cv2.cvtColor(img_with_mask_colors, cv2.COLOR_BGR2HSV)

    for result in results:
        boxes = result.boxes.xyxy
        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            width = x2 - x1
            height = y2 - y1
            center = (x1 + width // 2, y1 + height // 2)
            axes = (width // 2, height // 2)

            # Estrai l'area della bounding box
            roi = hsv_image[y1:y2, x1:x2]
            
            # Calcola il colore medio (Hue, Saturation, Value)
            mean_color = cv2.mean(roi)[:3]  # Solo i primi 3 canali (H, S, V)

            # Converti il colore medio in BGR per il riempimento
            mean_color_bgr = cv2.cvtColor(np.uint8([[mean_color]]), cv2.COLOR_HSV2BGR)[0][0]

            # Disegna l'ellisse con il colore medio
            cv2.ellipse(img_with_mask_colors, center, axes, 0, 0, 360, mean_color_bgr.tolist(), -1)

            # Disegna il contorno nero attorno all'ellisse
            cv2.ellipse(img_with_mask_colors, center, axes, 0, 0, 360, (0, 0, 0), 2)

    return img_with_mask_colors


# Funzione per classificare in scala di grigi le olive
def classify_olive_by_gray_intensity(gray_value):
    if gray_value <= 120:  # Valori di grigio alti -> Olives mature (nere)
        return "Ripe", (0, 200, 0)  # Verde
    elif 120 <= gray_value < 180:  # Valori di grigio intermedi -> Olives unripe (verdi)
        return "Unripe", (0, 165, 255)  # Arancione
    else:  # Valori di grigio bassi -> Olives semi mature (gialle)
        return "Partially ripe", (0, 255, 255)  # Giallo


# Funzione per riempire le olive con il colore del contorno
def classify_and_fill_colored_contours(image, gray_image, results):
    img_with_filled_ellipses = image.copy()
    mature_count, poco_mature_count, non_mature_count = 0, 0, 0
    
    for result in results:
        boxes = result.boxes.xyxy
        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            width = x2 - x1
            height = y2 - y1
            center = (x1 + width // 2, y1 + height // 2)
            axes = (width // 2, height // 2)

            # Extract the average grayscale intensity inside the ellipse
            mask = np.zeros(gray_image.shape, dtype=np.uint8)
            cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
            mean_gray_value = cv2.mean(gray_image, mask=mask)[0]

            # Classify the olive based on the grayscale intensity
            maturity_status, fill_color = classify_olive_by_gray_intensity(mean_gray_value)

            # Riempi l'ellisse con il colore
            cv2.ellipse(img_with_filled_ellipses, center, axes, 0, 0, 360, fill_color,-2)

            # Count the olives by maturity
            if maturity_status == 'Ripe':
                mature_count += 1
            elif maturity_status == 'Partially ripe':
                poco_mature_count += 1
            else:
                non_mature_count += 1

    return img_with_filled_ellipses, mature_count, poco_mature_count, non_mature_count


# Funzione principale per processare e salvare i risultati
def process_and_save_results(folder_path, upload_path):
    model = load_yolo_model()

    results_folder = os.path.join(folder_path, 'results')
    if not os.path.exists(results_folder):
        os.makedirs(results_folder)

    plots_folder = os.path.join(folder_path, 'plots')
    if not os.path.exists(plots_folder):
        os.makedirs(plots_folder)

    images_original = []
    images_boxes = []
    images_ellipses = []
    images_mask_colors = []
    images_maturity = []
    weights_all_images = []

    mature_total, poco_mature_total, non_mature_total = 0, 0, 0
    total_olives_sum = 0
    total_weight_sum = 0

    for filename in os.listdir(upload_path):
        if filename.endswith('.jpg') or filename.endswith('.png'):  
            image_path = os.path.join(upload_path, filename)
            print(f"Processando {image_path}...")

            # Creazione di una sottocartella per ogni immagine
            image_folder = os.path.join(results_folder, os.path.splitext(filename)[0])
            if not os.path.exists(image_folder):
                os.makedirs(image_folder)

            image = load_image(image_path)
            enhanced_image = enhance_image(image)
            results = run_yolo_inference(model, image_path, conf=0.5)

            img_boxes_conf, total_olives = draw_boxes_with_confidence(image, results)
            img_ellipses, total_weight, weights = draw_ellipses_and_weight(image, results)
            mask_colors = create_mask_with_olive_colors(enhanced_image, results)

            gray_image = cv2.cvtColor(enhanced_image, cv2.COLOR_BGR2GRAY)
            img_maturity, mature_count, poco_mature_count, non_mature_count = classify_and_fill_colored_contours(mask_colors, gray_image, results)

            # Salva le immagini nella sottocartella corrispondente
            cv2.imwrite(os.path.join(image_folder, f'{filename}_add_original.jpg'), image)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_boxes.jpg'), img_boxes_conf)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_ellipses.jpg'), img_ellipses)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_mask_colors.jpg'), mask_colors)
            cv2.imwrite(os.path.join(image_folder, f'{filename}_maturity.jpg'), img_maturity)

            # Aggiorna le liste per generare i grafici
            images_original.append(image)
            images_boxes.append((img_boxes_conf, total_olives))  # Include il numero di olive
            images_ellipses.append(img_ellipses)
            images_mask_colors.append(mask_colors)
            images_maturity.append(img_maturity)

            mature_total += mature_count
            poco_mature_total += poco_mature_count
            non_mature_total += non_mature_count
            total_olives_sum += total_olives
            total_weight_sum += total_weight
            weights_all_images.extend(weights)

            print(f"Completato {filename}")

    # Chart of the olive distribution by maturity
    categories = ['Mature', 'Partially ripe', 'Non mature']
    counts = [mature_total, poco_mature_total, non_mature_total]
    colors = ['lightgreen', 'yellow', 'orange']

    plt.figure(figsize=(8, 6))
    bars = plt.bar(categories, counts, color=colors, edgecolor='black')

    # Aggiungi il numero di olive sopra ciascuna barra
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{count}', ha='center', fontsize=10, color='black')

    plt.title('Olive distribution by maturity')
    plt.xlabel('Maturity level')
    plt.ylabel('Numero di olive')
    plt.savefig(os.path.join(plots_folder, 'olive_maturity_distribution.png'))  # Salva il grafico della distribuzione
    plt.close()  # Chiudi il grafico per evitare problemi con Tkinter

    # Istogramma della distribuzione dei pesi delle olive
    plt.figure(figsize=(8, 6))
    weights_hist, bins, patches = plt.hist(weights_all_images, bins=10, color='green', edgecolor='black')

    # Aggiungi il numero di olive sopra ciascuna barra dell'istogramma
    for i in range(len(patches)):
        height = weights_hist[i]
        plt.text(patches[i].get_x() + patches[i].get_width() / 2, height + 0.2,  # Aggiungi un piccolo offset per evitare la sovrapposizione
                f'{int(height)}', ha='center', va='bottom', fontsize=10, color='black')

    plt.title('Distribution del weight delle olive')
    plt.xlabel('Weight in grammi')
    plt.ylabel('Numero di olive')
    plt.savefig(os.path.join(plots_folder, 'olive_weight_distribution.png'))  # Salva il grafico della distribuzione dei pesi
    plt.close()  # Chiudi il grafico per evitare problemi con Tkinter

# Esegui la funzione
folder_path = r'C:\Users\Bartolo\OlivesDetectionProject\backend'
upload_path = os.path.join(folder_path, 'upload')
process_and_save_results(folder_path, upload_path)


Processando C:\Users\Bartolo\OlivesDetectionProject\backend\upload\0259823a-54ae-41bf-b677-0085400a6dd3_09b352d3-0f2a-40df-809b-6fc2f154a8ed_jpg.rf.7a9173fbc3ca414a944e3a791f06cbbe.jpg...

image 1/1 C:\Users\Bartolo\OlivesDetectionProject\backend\upload\0259823a-54ae-41bf-b677-0085400a6dd3_09b352d3-0f2a-40df-809b-6fc2f154a8ed_jpg.rf.7a9173fbc3ca414a944e3a791f06cbbe.jpg: 640x640 9 olives, 40.6ms
Speed: 5.0ms preprocess, 40.6ms inference, 169.9ms postprocess per image at shape (1, 3, 640, 640)
Completato 0259823a-54ae-41bf-b677-0085400a6dd3_09b352d3-0f2a-40df-809b-6fc2f154a8ed_jpg.rf.7a9173fbc3ca414a944e3a791f06cbbe.jpg
Processando C:\Users\Bartolo\OlivesDetectionProject\backend\upload\81cf257a-ccd1-4775-951a-aa1ce2c3e04d_6fff5ed4-a4ff-4ff7-92ef-cc67b6f96116_jpg.rf.025463641e5b985fae10482e77bbe0a7.jpg...

image 1/1 C:\Users\Bartolo\OlivesDetectionProject\backend\upload\81cf257a-ccd1-4775-951a-aa1ce2c3e04d_6fff5ed4-a4ff-4ff7-92ef-cc67b6f96116_jpg.rf.025463641e5b985fae10482e77bbe0a7.jpg: 6